<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/1_from_pdfs_to_text_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: From PDFs (and Other Files) to Analyzable Text

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

Every method in this course starts from clean text in a table. Real research material rarely
arrives that way. It comes as PDFs of policy reports, scanned archive documents, Word files from
interviewers, and web pages full of menus and adverts. Getting from those files to usable text is
often the most time-consuming part of a text-as-data project, and errors made here silently
affect everything downstream.

This notebook is a practical toolkit. You will:

- extract text from **born-digital PDFs** with three popular libraries, and fix the classic
  problems (two-column layouts, headers and footers, hyphenation)
- pull **tables** out of PDFs into pandas
- read **scanned** documents with **OCR** (optical character recognition), in English and Dutch
- read **Word** documents and **web pages**
- **measure** how accurate your extraction is
- combine everything into a **batch pipeline** that turns a folder of mixed files into one dataset

So that the notebook works anywhere, it starts by **generating its own test documents**. That has
a big advantage: we know exactly what text each document contains, so we can measure extraction
errors precisely. At the end you'll see how to run everything on your own files.

**Contents**

1. The document zoo
2. Making test documents
3. Born-digital PDFs: three libraries
4. Fixing the layout: columns, headers and footers
5. Tables
6. Measuring extraction quality
7. Cleaning extracted text
8. Scanned documents: OCR
9. Word documents and web pages
10. A batch pipeline: from a folder of files to a dataset
11. Your own files, and tools for harder cases

> No GPU needed. The setup cell installs a few libraries and the Tesseract OCR engine, which takes
> about a minute.

There are 5 exercises along the way, marked ✏️. A separate solutions notebook has worked answers, but try them yourself first.

## 0. Setup

We install:

- **pypdf**, **pdfplumber** and **PyMuPDF**: three libraries for reading PDFs
- **python-docx**: for Word files
- **reportlab**: to *create* PDFs (only needed for our test documents)
- **Tesseract** (with English and Dutch language data) and **pytesseract**: the OCR engine and its
  Python interface. Tesseract is a separate program, so it's installed with `apt-get`, Colab's
  system package manager.

In [ ]:
!pip install -q pypdf pdfplumber pymupdf python-docx reportlab pytesseract
!apt-get install -y -q tesseract-ocr tesseract-ocr-nld > /dev/null

import io
import os
import re
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import pypdf
import pdfplumber
import pymupdf                      # PyMuPDF (older code imports it as `fitz`)
import docx                         # python-docx
import pytesseract
from bs4 import BeautifulSoup

pd.set_option("display.max_colwidth", 120)
DOCS = Path("documents")
DOCS.mkdir(exist_ok=True)
print("Tesseract", pytesseract.get_tesseract_version(), "| languages:", pytesseract.get_languages())

## 1. The document zoo

| Kind of file | What's inside | How hard? | Main tools |
|---|---|---|---|
| **Born-digital PDF** (exported from Word, LaTeX, a website) | a *text layer*: the characters are stored, along with their positions on the page | easy to medium | pypdf, pdfplumber, PyMuPDF |
| **Scanned PDF** or photo | only *pictures* of pages; no characters at all | hard | OCR: Tesseract and others |
| **Word** (`.docx`) | structured text: paragraphs, headings, tables | easy | python-docx |
| **Web page** (HTML) | text mixed with menus, adverts, scripts | medium | BeautifulSoup, trafilatura |
| **Plain text** (`.txt`, `.csv`) | just text, but watch the *encoding* | easy | Python itself |

The single most important question for a PDF is: **does it have a text layer?** If you can select
and copy text in your PDF viewer, it probably does. If not, you need OCR.

A PDF doesn't store "paragraphs" or "columns". It stores instructions like *"draw these characters
at these coordinates"*. Reconstructing the reading order from those positions is where most
extraction problems come from.

## 2. Making test documents

We generate a small, realistic collection: a two-page **policy brief** in two columns, with a
running header, page numbers, hyphenated words and a table; a **scanned** copy of it (an image of
each page, slightly tilted and noisy, like a real scan); a short **Dutch** text, also as a scan; a
**Word** file; and a **web page**. You don't need to understand this code to follow the rest of
the notebook; just run it.

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib import colors
from reportlab.platypus import Table, TableStyle
from reportlab.pdfbase.pdfmetrics import stringWidth

TITLE = "Nitrogen, Farming and Public Health: What the Evidence Says"
PARAGRAPHS = [
    "Agricultural nitrogen emissions have become one of the most contested policy issues in the "
    "Netherlands. Ammonia from livestock farming contributes to the formation of fine particulate "
    "matter, which is associated with respiratory and cardiovascular conditions. At the same time, "
    "farming communities argue that proposed reductions threaten their livelihoods and the "
    "viability of family businesses that have existed for generations.",
    "Public opinion on the issue is sharply divided. Surveys show that a majority of citizens "
    "support measures to protect nature reserves, but trust in government institutions has "
    "declined among rural respondents. Communication about the scientific evidence plays a central "
    "role: how emissions, deposition and health effects are framed shapes whether policies are "
    "perceived as legitimate.",
    "This brief summarizes recent research on three questions. First, how large are the health "
    "effects attributable to agricultural emissions? Second, which policy instruments reduce "
    "emissions most efficiently? Third, how can governments communicate uncertainty without "
    "undermining public confidence in science?",
    "The evidence suggests that targeted measures near vulnerable areas deliver the largest "
    "benefits per euro spent. Voluntary buyout schemes have had limited uptake so far, whereas "
    "innovation subsidies for low-emission housing systems show promising results. Transparent "
    "communication, including openly acknowledging what is not yet known, appears to strengthen "
    "rather than weaken support for evidence-based policy.",
    "Three recommendations follow from this evidence. Policies should prioritize emission "
    "reductions around the most vulnerable nature areas, where the health and ecological benefits "
    "are largest. Support for farmers should combine financial compensation with practical help in "
    "adopting new technologies. And public communication should be consistent across ministries, "
    "explaining both the certainties and the open questions in plain language.",
]
# The exact text of the brief, for measuring extraction errors later
GROUND_TRUTH = TITLE + "\n\n" + "\n\n".join(PARAGRAPHS)          # whole document
PAGE1_TRUTH = TITLE + "\n\n" + "\n\n".join(PARAGRAPHS[:4])       # page 1 only
TABLE_ROWS = [["Policy area", "2024", "2025", "2026"],
              ["Nature restoration", "410", "455", "520"],
              ["Farm innovation", "180", "240", "310"],
              ["Public health monitoring", "35", "40", "48"],
              ["Communication", "12", "15", "15"]]


def wrap_with_hyphens(text, font, size, width):
    # Greedy line wrapping that splits long words with a hyphen, as typesetting software does
    lines, line = [], ""
    for word in text.split():
        candidate = (line + " " + word).strip()
        if stringWidth(candidate, font, size) <= width:
            line = candidate
            continue
        if len(word) >= 9:                                    # try to hyphenate long words
            for cut in range(len(word) - 3, 3, -1):
                head = (line + " " + word[:cut] + "-").strip()
                if stringWidth(head, font, size) <= width:
                    lines.append(head)
                    line = word[cut:]
                    break
            else:
                lines.append(line)
                line = word
        else:
            lines.append(line)
            line = word
    lines.append(line)
    return lines


def draw_page_furniture(c, page_no, n_pages):
    w, h = A4
    c.setFont("Helvetica", 8)
    c.setFillColor(colors.grey)
    c.drawString(50, h - 35, "POLICY BRIEF  |  No. 7  |  October 2026")
    c.drawRightString(w - 50, h - 35, "Example Institute for Rural Policy (fictional)")
    c.line(50, h - 40, w - 50, h - 40)
    c.drawCentredString(w / 2, 30, f"Page {page_no} of {n_pages}")
    c.setFillColor(colors.black)


def make_policy_brief(path):
    w, h = A4
    c = canvas.Canvas(str(path), pagesize=A4)
    draw_page_furniture(c, 1, 2)
    c.setFont("Helvetica-Bold", 15)
    c.drawString(50, h - 80, TITLE)

    font, size, leading = "Helvetica", 10.5, 14.5
    col_width = (w - 100 - 24) / 2
    for col, paras in enumerate([PARAGRAPHS[:2], PARAGRAPHS[2:4]]):   # two columns on page 1
        x, y = 50 + col * (col_width + 24), h - 115
        for para in paras:
            for line in wrap_with_hyphens(para, font, size, col_width):
                c.setFont(font, size)
                c.drawString(x, y, line)
                y -= leading
            y -= leading * 0.6                                  # space between paragraphs
    c.showPage()

    draw_page_furniture(c, 2, 2)                                # page 2: one column plus a table
    y = h - 80
    for line in wrap_with_hyphens(PARAGRAPHS[4], font, size, w - 100):
        c.setFont(font, size)
        c.drawString(50, y, line)
        y -= leading
    c.setFont("Helvetica-Bold", 10)
    c.drawString(50, y - 25, "Table 1. Public spending by policy area (million euros)")
    table = Table(TABLE_ROWS, colWidths=[200, 70, 70, 70])
    table.setStyle(TableStyle([("FONT", (0, 0), (-1, 0), "Helvetica-Bold"),
                               ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
                               ("ALIGN", (1, 0), (-1, -1), "RIGHT")]))
    tw, th = table.wrapOn(c, w, h)
    table.drawOn(c, 50, y - 35 - th)
    c.save()


make_policy_brief(DOCS / "policy_brief.pdf")
print("Created", DOCS / "policy_brief.pdf")

Let's look at what we made. PyMuPDF can render any PDF page as an image:

In [ ]:
def show_pages(path, dpi=60):
    with pymupdf.open(path) as pdf:
        images = [Image.open(io.BytesIO(page.get_pixmap(dpi=dpi).tobytes("png"))) for page in pdf]
    fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 7))
    for ax, im in zip(np.atleast_1d(axes), images):
        ax.imshow(im, cmap="gray"); ax.axis("off")
    plt.tight_layout()
    plt.show()

show_pages(DOCS / "policy_brief.pdf")

Now the **scanned** version. A scanner produces only an image of each page: we render the pages,
tilt them slightly, add some noise, and save the images as a new PDF. This PDF contains no text
at all, only pictures. We do the same for a short **Dutch** text.

In [ ]:
DUTCH_TEXT = ("De coördinatie tussen gemeenten en provincies is essentieel voor een efficiënt "
              "stikstofbeleid. Boeren, natuurorganisaties en overheden moeten samenwerken om de doelen "
              "voor 2030 te halen. Ook de volksgezondheid profiteert van schonere lucht, want fijnstof "
              "en ammoniak dragen bij aan luchtwegaandoeningen. Daarnaast is het vertrouwen in de "
              "wetenschap een belangrijke factor voor het draagvlak van maatregelen.")


def make_text_pdf(path, text, title):
    w, h = A4
    c = canvas.Canvas(str(path), pagesize=A4)
    c.setFont("Helvetica-Bold", 14)
    c.drawString(50, h - 70, title)
    y = h - 100
    for line in wrap_with_hyphens(text, "Helvetica", 11, w - 100):
        c.setFont("Helvetica", 11)
        c.drawString(50, y, line)
        y -= 15
    c.save()


def make_scan(pdf_path, scan_path, dpi=150, angle=0.8, noise=18, seed=0):
    # Turn every page into a slightly rotated, noisy grayscale image, and save those as a PDF
    rng = np.random.default_rng(seed)
    pages = []
    with pymupdf.open(pdf_path) as pdf:
        for page in pdf:
            im = Image.open(io.BytesIO(page.get_pixmap(dpi=dpi).tobytes("png"))).convert("L")
            im = im.rotate(angle, fillcolor=255)                       # a slightly crooked scan
            arr = np.array(im, dtype=float) + rng.normal(0, noise, (im.height, im.width))
            pages.append(Image.fromarray(np.clip(arr, 0, 255).astype("uint8")))
    pages[0].save(scan_path, save_all=True, append_images=pages[1:], resolution=dpi)


make_scan(DOCS / "policy_brief.pdf", DOCS / "policy_brief_scan.pdf")
make_text_pdf(DOCS / "stikstof_nl.pdf", DUTCH_TEXT, "Stikstof en gezondheid")
make_scan(DOCS / "stikstof_nl.pdf", DOCS / "stikstof_nl_scan.pdf")
os.remove(DOCS / "stikstof_nl.pdf")          # keep only the scanned Dutch version
show_pages(DOCS / "policy_brief_scan.pdf")

Finally a **Word** file and a **web page** with the same kind of content.

In [ ]:
word = docx.Document()
word.add_heading("Interview notes: farmer focus group", level=1)
word.add_paragraph("Participants described the nitrogen debate as a conflict between the city and the countryside.")
word.add_paragraph("Several participants said they no longer trust the figures published by the government.")
t = word.add_table(rows=1, cols=2)
t.rows[0].cells[0].text, t.rows[0].cells[1].text = "Participant", "Main concern"
for who, concern in [("P1", "future of the family farm"), ("P2", "unclear rules"), ("P3", "health of grandchildren")]:
    row = t.add_row()
    row.cells[0].text, row.cells[1].text = who, concern
word.save(DOCS / "focus_group.docx")

html = '''<html><head><title>Nitrogen plan presented</title>
<style>body { font-family: sans-serif; }</style><script>trackVisitor();</script></head>
<body>
<nav><a href="/">Home</a> | <a href="/news">News</a> | <a href="/sport">Sport</a> | <a href="/login">Log in</a></nav>
<div class="ad">Subscribe now for only 1 euro!</div>
<article>
  <h1>Government presents new nitrogen plan</h1>
  <p class="byline">By our political desk</p>
  <p>The cabinet presented its long-awaited plan to reduce nitrogen emissions on Friday.</p>
  <p>Farmers' organizations reacted critically, while environmental groups called the plan a first step.</p>
</article>
<footer>&copy; 2026 Example News (fictional). All rights reserved. Cookie settings.</footer>
</body></html>'''
(DOCS / "news_article.html").write_text(html, encoding="utf-8")

for f in sorted(DOCS.iterdir()):
    print(f"{f.name:<26} {f.stat().st_size / 1024:>7.1f} KB")

## 3. Born-digital PDFs: three libraries

### pypdf

The simplest option, written in pure Python.

In [ ]:
reader = pypdf.PdfReader(DOCS / "policy_brief.pdf")
print(f"{len(reader.pages)} pages\n")
text_pypdf = reader.pages[0].extract_text()
print(text_pypdf[:1500])

Look carefully, and you'll spot the classic problems:

- the **header** (*POLICY BRIEF | No. 7 ...*) and the **page number** are mixed into the text
- every line ends with a **line break**, wherever the layout happened to wrap
- words split across lines are still **hyphenated** (*Netherl-* / *ands*, *livel-* / *ihoods*)

The columns, on the other hand, come out in the right order. That's because pypdf follows the
order in which the text was *written into the file*, which for this PDF happens to be column by
column. In plenty of real PDFs it isn't, and then the text comes out scrambled.

None of these matter much to a human reader, but each one creates noise in a word count: *contrib*
and *utes* become two meaningless "words", and the header appears once per page in every document.

### pdfplumber

Built for precise layout work: it knows the position of every character, word and line, which
makes it the best choice for cropping and for tables.

In [ ]:
with pdfplumber.open(DOCS / "policy_brief.pdf") as pdf:
    page = pdf.pages[0]
    text_plumber = page.extract_text()
    words = page.extract_words()                     # every word with its coordinates

print(text_plumber[:700])
print("\nThe first five words, with their positions on the page (in points from the top left):")
pd.DataFrame(words[:5])[["text", "x0", "x1", "top", "bottom"]].round(1)

pdfplumber reads strictly line by line across the page, so with two columns, each line of the
left column gets glued to the line next to it in the right column. The result reads like
nonsense. We'll fix that with cropping in Section 4.

### PyMuPDF

Very fast, because it's built on a C library. Asking for `"blocks"` returns the text in layout
blocks (roughly: paragraphs), with their positions. With `sort=True`, PyMuPDF orders the blocks by
position: top to bottom, and left to right for blocks at the same height.

In [ ]:
with pymupdf.open(DOCS / "policy_brief.pdf") as pdf:
    page = pdf[0]
    text_pymupdf = page.get_text("text", sort=True)
    blocks = page.get_text("blocks", sort=True)      # (x0, y0, x1, y1, text, block_no, block_type)

print(f"{len(blocks)} blocks on page 1, in sorted order:\n")
for b in blocks:
    print(f"[x={b[0]:3.0f}, y={b[1]:3.0f}]  {b[4][:70]!r}")

Look at the order of the paragraph blocks (`x=50` is the left column, `x=310` the right one).
Sorting top to bottom **mixes the two columns**: a paragraph in the right column that starts
higher on the page than the next paragraph on the left jumps ahead of it, and blocks that start at
practically the same height can come out in either order. "Top to bottom, then left to right" is
the right rule for a single column and the wrong one for two. Without `sort=True`, PyMuPDF uses the order in the file, which for this PDF is
correct (Exercise 1). There's no setting that's right for every PDF, which is why you always look
at the output.

> **✏️ Exercise 1**
>
> 1. Time the three libraries on the full policy brief: extract all pages, 20 times each, and
>    report the average time per document. (Use `time.perf_counter()`.)
> 2. Print the full PyMuPDF text of page 1 with `get_text("text")` (without `sort=True`). Is the
>    reading order correct?

In [ ]:
# Your code here

## 4. Fixing the layout: columns, headers and footers

The most reliable fix for a known, consistent layout is **cropping**: tell pdfplumber exactly which
part of the page to read. A *bounding box* is `(x0, top, x1, bottom)` in points, measured from the
top-left corner (an A4 page is about 595 × 842 points).

We skip the header strip at the top and the page number at the bottom, and read the left column
before the right one.

In [ ]:
def extract_columns(path, header=50, footer=45, title_height=0, gap=12):
    # Read a two-column PDF in the right order, without header and footer; one text per page.
    # title_height: extra space to skip on page 1 (where the title spans both columns)
    pages = []
    with pdfplumber.open(path) as pdf:
        for i, page in enumerate(pdf.pages):
            out = []
            top = header + (title_height if i == 0 else 0)
            bottom = page.height - footer
            if i == 0:
                title = page.crop((0, header, page.width, top)).extract_text() or ""
                out.append(title.strip())
                mid = page.width / 2
                for x0, x1 in [(0, mid - gap / 2), (mid + gap / 2, page.width)]:   # left, then right
                    out.append(page.crop((x0, top, x1, bottom)).extract_text() or "")
            else:                                                                  # page 2: one column
                body = page.crop((0, top, page.width, bottom))
                table_top = min((t.bbox[1] for t in body.find_tables()), default=bottom)
                out.append(body.crop((0, top, page.width, table_top - 25)).extract_text() or "")
            pages.append("\n".join(out))
    return pages


column_pages = extract_columns(DOCS / "policy_brief.pdf", title_height=50)
text_columns = "\n".join(column_pages)
print(text_columns)

Header and page numbers are gone, the columns are in the right order, and on page 2 we stopped
before the table (which we'll extract properly in a moment). What's left is line breaks and
hyphenation, which Section 7 cleans up.

The downside of cropping: the coordinates are specific to one layout. That's fine for a series of
reports from the same organization, which tend to share a template, but you'll need to check them
for each new source. Always render a page or two and look at it.

## 5. Tables

Tables are where plain text extraction really falls apart: each cell becomes a loose word.
pdfplumber can detect tables from their lines and spacing and return them as rows.

In [ ]:
with pdfplumber.open(DOCS / "policy_brief.pdf") as pdf:
    tables = pdf.pages[1].extract_tables()

print(f"Found {len(tables)} table(s) on page 2")
spending = pd.DataFrame(tables[0][1:], columns=tables[0][0])
spending[["2024", "2025", "2026"]] = spending[["2024", "2025", "2026"]].astype(int)
spending

That's a proper DataFrame with numbers you can calculate with. Table detection works best when
tables have visible lines, as here; tables laid out with spaces only are harder (pdfplumber's
`table_settings` let you tune the detection, and the tools in Section 11 do better on complex
tables).

> **✏️ Exercise 2**
>
> Using the `spending` table, compute the growth in spending between 2024 and 2026 for each policy
> area, in percent, and make a horizontal bar chart of it.

In [ ]:
# Your code here

## 6. Measuring extraction quality

Since we know the true text of our documents, we can measure how far an extraction is from it.
The standard measure is the **character error rate** (CER): the minimum number of characters you
would have to insert, delete or change to turn the extracted text into the true text (the *edit
distance*), divided by the length of the true text. A CER of 0.02 means 2 errors per 100
characters.

Before comparing, we normalize whitespace on both sides: we care about the characters, not about
where lines happened to break.

In [ ]:
def edit_distance(a, b):
    # Levenshtein distance: minimum number of insertions, deletions and substitutions
    if len(a) < len(b):
        a, b = b, a
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        current = [i]
        for j, cb in enumerate(b, start=1):
            current.append(min(previous[j] + 1,                  # deletion
                               current[j - 1] + 1,               # insertion
                               previous[j - 1] + (ca != cb)))    # substitution (free if equal)
        previous = current
    return previous[-1]


def normalize_ws(text):
    return re.sub(r"\s+", " ", text).strip()


def cer(extracted, truth):
    extracted, truth = normalize_ws(extracted), normalize_ws(truth)
    return edit_distance(extracted, truth) / len(truth)


print("edit_distance('kitten', 'sitting') =", edit_distance("kitten", "sitting"), "(the textbook example: 3)")

We compare three extractions with the true text of the brief (title and paragraphs). The
whole-page extractions also contain the header, the page numbers and the table, and those count as
errors, which is fair: in a word count they *are* noise. For pdfplumber, the scrambled columns
add many more errors.

In [ ]:
text_pypdf_all = "\n".join(p.extract_text() for p in pypdf.PdfReader(DOCS / "policy_brief.pdf").pages)
with pdfplumber.open(DOCS / "policy_brief.pdf") as pdf:
    text_plumber_all = "\n".join(p.extract_text() for p in pdf.pages)

print(f"pypdf, whole pages:            CER {cer(text_pypdf_all, GROUND_TRUTH):.3f}")
print(f"pdfplumber, whole pages:       CER {cer(text_plumber_all, GROUND_TRUTH):.3f}")
print(f"pdfplumber, cropped columns:   CER {cer(text_columns, GROUND_TRUTH):.3f}")

## 7. Cleaning extracted text

A small cleaning function fixes most of what remains. Each step is one line, and each one is
worth understanding, because each can also do damage if your documents are different:

In [ ]:
def clean_extracted(text):
    text = unicodedata.normalize("NFKC", text)              # ligatures like "ﬁ" -> "fi", odd spaces -> " "
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)           # re-join words hyphenated at a line break
    text = re.sub(r"[ \t]+\n", "\n", text)                  # trailing spaces at line ends
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)             # single line break -> space (keep paragraph breaks)
    text = re.sub(r"[ \t]{2,}", " ", text)                   # repeated spaces
    return text.strip()


cleaned = clean_extracted(text_columns)
print(cleaned[:800])
print(f"\nCER before cleaning: {cer(text_columns, GROUND_TRUTH):.3f}")
print(f"CER after cleaning:  {cer(cleaned, GROUND_TRUTH):.3f}")

A word of caution about the hyphen rule: it can't tell a line-break hyphen (*contrib-utes*) from a
real one that happens to fall at the end of a line (*evidence-based*). Our rule turns both into one
word, so *evidence-based* would become *evidencebased*. For most word-count analyses that's an
acceptable trade-off, but if hyphenated compounds matter for your research, check a sample by hand.

About **ligatures**: typeset documents (LaTeX, professional layouts) often store "fi", "fl" and
"ff" as single special characters. They look normal on screen, but the computer sees a different
character, so *ﬁnancial* won't match *financial* in your dictionary. NFKC normalization fixes it:

In [ ]:
looks_normal = "the \ufb01nancial e\ufb00ects of \ufb02ooding"     # what you often get from typeset PDFs
print(looks_normal, "→ contains 'financial'?", "financial" in looks_normal)
print(unicodedata.normalize("NFKC", looks_normal), "→ after NFKC:", "financial" in unicodedata.normalize("NFKC", looks_normal))

> **✏️ Exercise 3**
>
> Often you can't crop, for example when documents come from many different sources. Write a
> function `remove_running_lines(pages)` that takes a list of page texts and removes every line
> that appears on (almost) **every** page, such as headers, and every line that consists only of a
> page-number pattern like `Page 2 of 7`. Apply it to the pypdf output of all pages, then clean
> it, and compare the CER with the uncleaned pypdf output.
>
> *Hint: count in how many pages each (stripped) line occurs with `collections.Counter`.*

In [ ]:
# Your code here

## 8. Scanned documents: OCR

### First, check whether a PDF needs OCR

A scanned PDF has no text layer, so extraction returns (almost) nothing. That's an easy test to
automate:

In [ ]:
def has_text_layer(path, min_chars=50):
    with pymupdf.open(path) as pdf:
        return sum(len(page.get_text().strip()) for page in pdf) >= min_chars

for f in ["policy_brief.pdf", "policy_brief_scan.pdf", "stikstof_nl_scan.pdf"]:
    print(f"{f:<24} text layer: {has_text_layer(DOCS / f)}")

### Running Tesseract

OCR works on images, so we render each page as an image first. Resolution matters: Tesseract works
best at around **300 dpi** (dots per inch). Then `pytesseract.image_to_string` returns the text.

In [ ]:
def page_images(path, dpi=300):
    with pymupdf.open(path) as pdf:
        return [Image.open(io.BytesIO(p.get_pixmap(dpi=dpi).tobytes("png"))) for p in pdf]


def ocr_pdf(path, lang="eng", dpi=300):
    return "\n".join(pytesseract.image_to_string(im, lang=lang) for im in page_images(path, dpi))


start = time.time()
text_ocr = ocr_pdf(DOCS / "policy_brief_scan.pdf")
print(f"OCR took {time.time() - start:.1f} s\n")
print(text_ocr[:900])

Three observations. OCR is **slow**: seconds per page, compared with milliseconds for a text
layer. Tesseract does its own **layout analysis**, and handles our two columns. And the tilted
header with its horizontal rule causes trouble: parts of it go missing or turn into stray
characters, a typical OCR problem around lines, logos and stamps. Look closely at the body text
too: you may spot misread letters, such as an `l` read as `!` or `t`.

To measure the OCR itself, we crop the page image to the body text, just as we cropped the text
layer in Section 4, and compare page 1 with its true text. PDF coordinates are in points (1/72
inch), so at 300 dpi we multiply them by 300/72 to get pixels.

In [ ]:
def ocr_region(path, page_no=0, box=(0, 50, 595, 797), lang="eng", dpi=300, prepare=None):
    # OCR one page, cropped to box = (x0, top, x1, bottom) in PDF points
    image = page_images(path, dpi)[page_no]
    scale = dpi / 72
    image = image.crop(tuple(int(v * scale) for v in box))
    if prepare is not None:                       # optional image preprocessing (Exercise 4)
        image = prepare(image)
    return pytesseract.image_to_string(image, lang=lang)


ocr_page1 = ocr_region(DOCS / "policy_brief_scan.pdf")
print(f"OCR of the scan, page 1 body, cleaned:  CER {cer(clean_extracted(ocr_page1), PAGE1_TRUTH):.3f}")
print(f"Text layer, page 1 body, cleaned:       CER {cer(clean_extracted(column_pages[0]), PAGE1_TRUTH):.3f}")

On a clean, synthetic scan like ours, Tesseract gets close to the text layer. Real scans, with
yellowed paper, faint print, stamps or typewriter fonts, are often much worse. Always measure on a
sample of your own documents.

### How sure is Tesseract?

`image_to_data` returns every recognized word with a **confidence** score from 0 to 100. Low
confidence flags the passages worth checking by hand, which is a cheap way to target your quality
control.

In [ ]:
ocr_words = pytesseract.image_to_data(page_images(DOCS / "policy_brief_scan.pdf")[0],
                                      output_type=pytesseract.Output.DATAFRAME)
ocr_words = ocr_words[(ocr_words["conf"] >= 0) & ocr_words["text"].notna()]
ocr_words = ocr_words[ocr_words["text"].astype(str).str.strip() != ""]
print(f"{len(ocr_words)} words, median confidence {ocr_words['conf'].median():.0f}")
print("\nThe least confident words:")
ocr_words.sort_values("conf")[["text", "conf"]].head(8)

> **✏️ Exercise 4**
>
> 1. Run `ocr_region` on page 1 of the scanned brief at 100, 200 and 300 dpi, and report the CER
>    against `PAGE1_TRUTH` (after cleaning) and the time for each. What's the trade-off?
> 2. Preprocessing sometimes helps. Convert the image to pure black and white with a threshold
>    before OCR, by passing `prepare=lambda im: im.convert("L").point(lambda v: 255 if v > 150 else 0)`.
>    Does it change the CER at 300 dpi?

In [ ]:
# Your code here

### Other languages

Tesseract needs a language model for each language: it uses it to recognize characters (such as
*ë* and *ö*) and to prefer real words when letters are ambiguous. Here's our Dutch scan, read with
the English and with the Dutch model:

In [ ]:
dutch_truth = "Stikstof en gezondheid\n\n" + DUTCH_TEXT
for lang in ["eng", "nld"]:
    t = clean_extracted(ocr_pdf(DOCS / "stikstof_nl_scan.pdf", lang=lang))
    print(f"lang={lang}: CER {cer(t, dutch_truth):.3f}")
    print("   ", t[:160], "...\n")

Check the words with diacritics in both versions, such as *coördinatie* and *efficiënt*. Several
languages can be combined, e.g. `lang="nld+eng"`, for documents that mix them. Tesseract supports
more than 100 languages; install others with `apt-get install tesseract-ocr-deu` (German),
`-fra` (French), and so on.

## 9. Word documents and web pages

### Word (`.docx`)

A `.docx` file stores real structure, so extraction is clean: paragraphs, their style (heading or
normal text), and tables.

In [ ]:
d = docx.Document(DOCS / "focus_group.docx")
for p in d.paragraphs:
    print(f"[{p.style.name}] {p.text}")

rows = [[cell.text for cell in row.cells] for row in d.tables[0].rows]
pd.DataFrame(rows[1:], columns=rows[0])

Older Word files (`.doc`, before 2007) use a different format that python-docx can't read. Convert
them first, for example with LibreOffice (free), which can also run from the command line:
`soffice --headless --convert-to docx myfile.doc`.

### Web pages

For HTML, the challenge is the opposite of PDFs: the text is all there, but surrounded by things
that aren't content. Grabbing all text gives you menus, adverts and cookie notices:

In [ ]:
soup = BeautifulSoup((DOCS / "news_article.html").read_text(encoding="utf-8"), "html.parser")
print("All text on the page:\n", soup.get_text(" ", strip=True), "\n")

for tag in soup(["script", "style", "nav", "footer"]):          # remove non-content elements
    tag.decompose()
for ad in soup.select(".ad, .byline"):                         # and elements by their CSS class
    ad.decompose()

article = soup.find("article")
print("Headline:", article.find("h1").get_text(strip=True))
print("Body:    ", " ".join(p.get_text(strip=True) for p in article.find_all("p")))

This works when you know the structure of a website (every site uses different tags and classes;
right-click → *Inspect* in your browser shows them). For collecting articles from many different
news sites, a library like **trafilatura** detects the main text automatically and is usually
the better choice. Always check a site's terms of use and `robots.txt` before collecting from it
at scale.

## 10. A batch pipeline: from a folder of files to a dataset

In a real project you have hundreds or thousands of files. Here is a pipeline that walks through a
folder, picks the right method for each file, records what it did, and produces **one table**
with a row per page (or per document), ready for the methods from this course.

Two principles make such a pipeline trustworthy:

- **Record the method** for every row (text layer or OCR), so you can check OCR rows separately.
- **Never let one bad file stop the run.** Catch errors, record them, and move on.

In [ ]:
def extract_file(path):
    # Extract text from one file; returns a list of dicts (one per page or per document)
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        with pymupdf.open(path) as pdf:
            needs_ocr = sum(len(p.get_text().strip()) for p in pdf) < 50
            rows = []
            for i, page in enumerate(pdf, start=1):
                if needs_ocr:
                    img = Image.open(io.BytesIO(page.get_pixmap(dpi=300).tobytes("png")))
                    text, method = pytesseract.image_to_string(img, lang="eng+nld"), "ocr"
                else:                                # file order; check a sample (Section 3)
                    text, method = page.get_text("text"), "text layer"
                rows.append({"page": i, "method": method, "text": clean_extracted(text)})
            return rows
    if suffix == ".docx":
        text = "\n\n".join(p.text for p in docx.Document(path).paragraphs if p.text.strip())
        return [{"page": None, "method": "docx", "text": text}]
    if suffix in (".html", ".htm"):
        soup = BeautifulSoup(path.read_text(encoding="utf-8", errors="replace"), "html.parser")
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()
        return [{"page": None, "method": "html", "text": soup.get_text(" ", strip=True)}]
    raise ValueError(f"unsupported file type: {suffix}")


def extract_folder(folder):
    rows, problems = [], []
    for path in sorted(Path(folder).iterdir()):
        try:
            for r in extract_file(path):
                rows.append({"file": path.name, **r, "n_chars": len(r["text"])})
        except Exception as e:                       # record the problem and continue
            problems.append({"file": path.name, "error": f"{type(e).__name__}: {e}"})
    return pd.DataFrame(rows), pd.DataFrame(problems)


(DOCS / "notes.xyz").write_text("a file type our pipeline doesn't know")    # to test error handling

start = time.time()
corpus, problems = extract_folder(DOCS)
print(f"Processed the folder in {time.time() - start:.1f} s")
display(corpus[["file", "page", "method", "n_chars", "text"]])
display(problems)

From here, the corpus is a normal DataFrame: save it (`corpus.to_csv(...)` or, better for large
corpora, `corpus.to_parquet(...)`) and continue with any method from the course: a document-term
matrix, embeddings, BERT.

> **✏️ Exercise 5**
>
> Extend `extract_file` so that it also handles plain text files (`.txt`). Watch out for the
> **encoding**: try UTF-8 first, and if that fails with a `UnicodeDecodeError`, fall back to
> `"cp1252"` (the old Windows encoding, common in files from older systems). Test it with two
> files: one saved as UTF-8 and one saved with `encoding="cp1252"`, both containing the word
> *coördinatie*.

In [ ]:
# Your code here

## 11. Your own files, and tools for harder cases

### Running the pipeline on your own documents

In Colab, upload files directly, or connect your Google Drive (better for many files: uploads
disappear when the session ends). Uncomment one of the two options:

In [ ]:
# Option A: upload a few files from your computer
# from google.colab import files
# MY_DOCS = Path("my_documents"); MY_DOCS.mkdir(exist_ok=True)
# for name, content in files.upload().items():
#     (MY_DOCS / name).write_bytes(content)

# Option B: use a folder on your Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# MY_DOCS = Path("/content/drive/MyDrive/my_project/documents")

# Then:
# my_corpus, my_problems = extract_folder(MY_DOCS)
# my_corpus.to_csv("my_corpus.csv", index=False)

> ⚠️ **Privacy.** Interview transcripts, internal reports and archive material are often
> confidential. Colab runs on Google's servers, so check your data management plan before
> uploading such files. Everything in this notebook also runs on your own computer (see the
> *intro-to-python* materials for setting up Python locally).

### When the basic tools aren't enough

| Problem | Tools worth knowing |
|---|---|
| Complex layouts, tables without lines, you want Markdown with headings | **Docling** (IBM), **Marker**, **PyMuPDF4LLM** |
| Scientific articles: title, authors, sections, references | **GROBID** |
| Poor-quality scans, unusual scripts | **EasyOCR**, **PaddleOCR** |
| Handwritten or historical documents | **Transkribus** (widely used in the humanities) |
| Very messy material | vision-language models, which "read" a page image (see below) |

Large vision-language models (the multimodal LLMs of Day 4) are increasingly good at reading
difficult pages, including tables and handwriting. Two warnings, though. They can **hallucinate**:
produce fluent text that isn't on the page, which classic OCR never does. And online models mean
sending your documents to a company. Run them locally where you can, and as always, measure the
error rate on a hand-checked sample before trusting them with a whole archive.

### A checklist for any extraction project

1. Look at a sample of your files first: born-digital or scanned? Layouts? Languages? Tables?
2. Pick the method per file type, and **record the method** for every document.
3. Hand-check a random sample of pages against the originals; compute the CER if you can.
4. Clean conservatively, and keep the raw extraction too, so you can redo the cleaning later.
5. Report the extraction process and its error rate in your paper's methods section.